In [ ]:
!pip install --upgrade ipython
!pip install -r ../

: 

In [7]:
import os
import sys
from google.colab import drive
from google.colab import auth
auth.authenticate_user()

# Hot Reload
# %load_ext autoreload
# %autoreload 2

# Anything related to the drive
drive.mount('/content/drive', force_remount=True)
PROJECT_ROOT = "/content/drive/MyDrive/"
SHARED_DATA_SHORTCUT = "/content/drive/MyDrive/gsl/GSL Dataset" 

if os.path.exists(PROJECT_ROOT):
    %cd {PROJECT_ROOT}
    print(f"Success: Currently in {os.getcwd()}")
else:
    print(f"Error: Folder not found at {PROJECT_ROOT}. Check your naming!")

print("\n--- Verifying Dataset Path ---")
if os.path.exists(SHARED_DATA_SHORTCUT):
    print(f"Dataset found! Contents:")
    print(os.listdir(SHARED_DATA_SHORTCUT)[:5])
else:
    print(f"Dataset shortcut NOT found at {SHARED_DATA_SHORTCUT}")
    print("Check if you added the 'Shortcut to My Drive' correctly.")

# This allows 'import gsl_detection' to work
if os.path.join(PROJECT_ROOT, "src") not in sys.path:
    sys.path.append(os.path.join(PROJECT_ROOT, "src"))

def run_smart_batch(batch_size=10):
    """Moves small batches to SSD, processes, and deletes to save space."""
    
    # Get list of all videos in the shared Drive folder
    all_vids = [f for f in os.listdir(SHARED_DATA_SHORTCUT) if f.endswith(('.mp4', '.avi'))]
    processed_file = os.path.join(PROJECT_ROOT, "data/processed/landmarks.npy")
    
    print(f"Total videos found: {len(all_vids)}")

    for i in range(0, len(all_vids), batch_size):
        batch = all_vids[i : i + batch_size]
        local_paths = []

        print(f"\nProcessing Batch {i//batch_size + 1} ({i} to {i+batch_size})...")

        # Step A: Copy batch to local SSD
        for vid in batch:
            src = os.path.join(SHARED_DATA_SHORTCUT, vid)
            dst = f"/content/{vid}"
            shutil.copy(src, dst)
            local_paths.append(dst)

        # Step B: Run the worker script
        # We use --append True so it doesn't overwrite previous batches
        !python -m gsl_detection.features.build_features \
            --input "/content" \
            --output "{processed_file}" \
            --append True

        # Step C: Clean up SSD to make room for next batch
        for path in local_paths:
            if os.path.exists(path):
                os.remove(path)
        
        print(f"🧹 SSD Cleared. Space remaining:")
        !df -h /content | grep '/content'

# 3. Start the process
run_smart_batch(batch_size=15) # Change 15 to 5 if videos are very large

Mounted at /content/drive
/content/drive/MyDrive
Success: Currently in /content/drive/MyDrive

--- Verifying Dataset Path ---
Dataset found! Contents:
['GSL_isolated.tar.gz', 'GSL_continuous.tar.gz', 'GSL_iso_files', 'GSL_continuous_files']
Total videos found: 0
